# 02 · Cosmology

Along a photon path the scale factor obeys (review notes, Eq. `dadx_general`)

$$
\frac{da}{dx} = \frac{H_0}{c}\,\frac{\sqrt{\Omega_r + \Omega_m a + \Omega_k a^2 + \Omega_\Lambda a^4}}{\sqrt{1 - k x^2}} .
$$

`Cosmology` holds a batch of backgrounds and integrates this equation with `diffrax`. The skewers (03) use it to solve
$a(x)$ along every path. Runtime: ~10 s.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)

In [ ]:
plot_dag(('cosmology',)); plt.show()

In [ ]:
from skewerjax.core import Cosmology, phys
from skewerjax.utils.plotting import plot_scale_factor_solutions
h = phys.H0_over_c                                                  # H0 / c [1/Mpc]
names = ["radiation", "matter", "Lambda", "Planck 2018"]
cosmo = Cosmology(H0=jnp.full(4, phys.H0), Omega_r_0=jnp.array([1.0, 0.0, 0.0, 9.1436e-5]), Omega_m_0=jnp.array([0.0, 1.0, 0.0, 0.3089]),
                  Omega_k_0=jnp.zeros(4), Omega_lambda_0=jnp.array([0.0, 0.0, 1.0, 0.6911]))
x0, a0 = 0.0, 1.0                                                   # initial conditions: the observer today
x_f, x_b = np.linspace(x0, x0 + 1 / (a0 * h) - 1e-6, 600), np.linspace(x0, x0 - 2 * np.sqrt(a0) / h + 1e-6, 600)
a = np.concatenate([np.asarray(cosmo.solve_a_of_x(jnp.full(4, a0), jnp.asarray(x_b)))[:, ::-1],
                    np.asarray(cosmo.solve_a_of_x(jnp.full(4, a0), jnp.asarray(x_f)))], 1)
x = np.concatenate([x_b[::-1], x_f])
exact = {"radiation": a0 + h * (x - x0),                             # da/dx = h
         "matter": (np.sqrt(a0) + 0.5 * h * (x - x0)) ** 2,          # da/dx = h a^(1/2)
         "Lambda": a0 / (1 - a0 * h * (x - x0)), "Planck 2018": None} # da/dx = h a^2
colors = dict(zip(names, [TOL["red"], TOL["blue"], TOL["green"], FG]))
fig, axs = plot_scale_factor_solutions(x, dict(zip(names, a)), exact, colors, x0=x0, a0=a0, h=h)
add_panel_letters(axs)
fig.suptitle("a(x) along a light ray: the ODE solution on the closed forms (thick, light), and where each one ends", fontsize=22)
plt.show()
for name, xs in (("radiation", x0 - a0 / h), ("matter", x0 - 2 * np.sqrt(a0) / h), ("Lambda", x0 + 1 / (a0 * h))):
    i = names.index(name)
    near = np.argmin(np.abs(x - xs))
    print(f"{name:9s}: special point x = {xs:9.1f} Mpc; numerical a there = {a[i, near]:.3e}; "
          f"max |a / exact - 1| = {np.nanmax(np.abs(a[i][exact[name] > 0] / exact[name][exact[name] > 0] - 1)):.1e}")

*Reading the figure.* With a single component, $da/dx = (H_0/c)\sqrt{\Omega\, a^p}$ has closed forms. The initial
condition is $a_0$ at $x_0$ (the observer today), marked by the cross and the dotted guide lines.

- **Radiation** ($p = 0$): $a = a_0 + (H_0/c)(x - x_0)$ vanishes at $x = x_0 - a_0 c/H_0$, the big bang along this ray.
- **Matter** ($p = 1$): $a = \big[a_0^{1/2} + \tfrac12 (H_0/c)(x - x_0)\big]^2$ reaches $a = a' = 0$ at
  $x = x_0 - 2 a_0^{1/2} c/H_0$.
- **$\Lambda$** ($p = 4$): $a = a_0/\big[1 - a_0 (H_0/c)(x - x_0)\big]$ diverges at $x = x_0 + c/(a_0 H_0)$, a finite
  comoving distance into the future.

The vertical lines mark these points in each background's colour; Planck 2018 lies between them. The residual (b) is
on a symmetric log scale, linear below $10^{-5}$, so the ODE error of every background is visible. It stays near the
solver tolerance and grows only next to the special points, where $a \to 0$ or $a \to \infty$.

The default tolerances (`rtol = atol = 1e-4`) give $a(x)$ to $\sim 10^{-4}$, and `solve_a_of_x(..., rtol=1e-10, atol=1e-12)`
gives it to $\sim 10^{-9}$ (`scripts/validation/check_cosmology.py`). The parameters are pytree leaves, so `jax.grad`
of any result with respect to $\Omega_m$ works.